In [ ]:
Content
1. Cheet Sheet

In [ ]:
+--------------------+-------------------------------------+----------------------------------------------------+
| Query Type         | Purpose                             | Example                                            |
+--------------------+-------------------------------------+----------------------------------------------------+
| SELECT Basic       | Retrieve specific columns/rows      | SELECT name, age FROM users WHERE age > 25;        |
| SELECT DISTINCT    | Remove duplicate results            | SELECT DISTINCT city FROM customers;               |
| WHERE Filtering    | Filter rows with conditions         | SELECT * FROM orders WHERE amount > 1000;          |
| ORDER BY           | Sort results                        | SELECT * FROM products ORDER BY price DESC;        |
| LIMIT / OFFSET     | Paginate results                    | SELECT * FROM logs LIMIT 10 OFFSET 20;             |
| BETWEEN / IN       | Range or set match                  | SELECT * FROM sales WHERE date BETWEEN '2024-01-01'|
|                    |                                     | AND '2024-12-31';                                  |
| LIKE / ILIKE       | Pattern matching                    | SELECT * FROM users WHERE name LIKE 'A%'; (ILike: Case-Insensitive)          |
| JOIN (INNER)       | Match rows in both tables           | SELECT u.name, o.id FROM users u                   |
|                    |                                     | JOIN orders o ON u.id = o.user_id;                 |
| LEFT JOIN          | Keep all from left table            | SELECT u.name, o.id FROM users u                   |
|                    |                                     | LEFT JOIN orders o ON u.id = o.user_id;            |
| RIGHT JOIN         | Keep all from right table           | SELECT * FROM a RIGHT JOIN b ON a.id = b.id;       |
| FULL OUTER JOIN    | Keep all from both                  | SELECT * FROM a FULL JOIN b ON a.id = b.id;        |
| GROUP BY           | Aggregate rows                      | SELECT city, COUNT(*) FROM customers GROUP BY city;|
| HAVING             | Filter aggregates                   | SELECT city, COUNT(*) FROM customers               |
|                    |                                     | GROUP BY city HAVING COUNT(*) > 5;                 |
| SUBQUERY           | Query inside a query                | SELECT name FROM users WHERE id IN                 |
|                    |                                     | (SELECT user_id FROM orders);                      |
| CTE (WITH)         | Temporary named result set          | WITH sales_cte AS (SELECT * FROM sales             |
|                    |                                     | WHERE amount > 1000) SELECT * FROM sales_cte;      |
| WINDOW FUNCTIONS   | Row-based calculations              | SELECT name, RANK() OVER (ORDER BY score DESC)     |
|                    |                                     | FROM students;                                     |
| CASE Expression    | Conditional value                   | SELECT name, CASE WHEN age >= 18 THEN 'Adult'      |
|                    |                                     | ELSE 'Minor' END FROM users;                       |
| INSERT             | Add new data                        | INSERT INTO users(name, age) VALUES ('A', 25);     |
| UPDATE             | Modify data                         | UPDATE users SET age = 30 WHERE id = 1;            |
| DELETE             | Remove data                         | DELETE FROM users WHERE id = 1;                    |
| UPSERT / MERGE     | Insert or update                    | INSERT ... ON DUPLICATE KEY UPDATE ...;            |
| INDEX HINTS        | Force index use                     | SELECT * FROM orders USE INDEX(order_idx) ...;     |
| ANALYZE / EXPLAIN  | Query execution plan                | EXPLAIN ANALYZE SELECT * FROM big_table;           |
| TRANSACTIONS       | Commit/rollback atomic ops          | BEGIN; UPDATE a; INSERT INTO b; COMMIT;            |
| LOCKING            | Manage concurrency                  | SELECT * FROM table FOR UPDATE;                    |
| PIVOT / UNPIVOT    | Reshape data                        | (varies per DB)                                    |
+--------------------+-------------------------------------+----------------------------------------------------+

In [ ]:
# Query 1: 

SELECT *
FROM Customers c
JOIN Orders o ON c.CustomerId = o.CustomerId
WHERE c.Country = 'India'
  AND o.TotalAmount > 10000

# Execution
1. Index Seek on Customers (Country = 'India') → 100k rows
2. Index Seek on Orders (TotalAmount > 10000) → 1M rows
3. Nested Loop Join on CustomerId → Output 10k rows
4. Apply residual filter: c.Name LIKE 'A%'

JOIN is not always performed first — it’s re-ordered.
The WHERE filters are applied before or during join whenever possible.

In [ ]:
# FAQ
1. find customers with no orders → LEFT JOIN + WHERE orders.id IS NULL.

In [ ]:
# Partition Query
 - GROUP BY → collapses rows
 - PARTITION BY → keeps rows, adds grouping logic


# Partition Query Types
ROW_NUMBER → unique ranks (no ties)
RANK       → skips numbers (1,1,3)
DENSE_RANK → no gaps (1,1,2)


# Example of ROW_NUMBER

SELECT *,
       ROW_NUMBER() OVER (
           PARTITION BY country 
           ORDER BY total_spent DESC
       ) AS rnk
FROM user_order_stats

id | name | country | total_spent | rnk
----------------------------------------
1  | A    | India   | 2000        | 1
2  | B    | India   | 1500        | 2
3  | C    | India   | 1200        | 3
4  | D    | US      | 3000        | 1
5  | E    | US      | 1000        | 2

In [ ]:
# Queries
1. Second Highest Salary
    select max(salary) 
        from employees 
        where salary < (select max(salary) from employee);


2. Nth Highest Salary
    SELECT salary
        FROM (
            SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
            FROM employees
        ) t
        WHERE rnk = N;


3. Find Duplicate Records
    SELECT email, COUNT(*)
        FROM users
        GROUP BY email
        HAVING COUNT(*) > 1;

4. Delete Duplicate Records
    DELETE FROM users
        WHERE id NOT IN (
            SELECT MIN(id)
            FROM users
            GROUP BY email
        );


5.  Employees Earning More Than Their Manager
    SELECT e.name
        FROM employees e
        JOIN employees m
        ON e.manager_id = m.id
        WHERE e.salary > m.salary;

6. Find missing number
SELECT n
    FROM generate_series(1, 100) n
    LEFT JOIN numbers t ON n = t.id
    WHERE t.id IS NULL;


7. Latest Record Per Group
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) AS rnk
    FROM orders
) t
WHERE rnk = 1;



8. 
Find top 5 users (per country) who:
    - Have placed at least 3 orders
    - Total order amount > 1000
    - Show their total spend and last order date

WITH user_order_stats AS (
    SELECT 
        u.id,
        u.name,
        u.country,
        COUNT(o.id) AS total_orders,
        SUM(o.amount) AS total_spent,
        MAX(o.created_at) AS last_order_date
    FROM Users u
    JOIN Orders o ON u.id = o.user_id
    WHERE o.created_at >= '2023-01-01'
    GROUP BY u.id, u.name, u.country
    HAVING COUNT(o.id) >= 3
       AND SUM(o.amount) > 1000
),

ranked_users AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY country 
               ORDER BY total_spent DESC
           ) AS rnk
    FROM user_order_stats
)

SELECT 
    id,
    name,
    country,
    total_orders,
    total_spent,
    last_order_date
FROM ranked_users
WHERE rnk <= 5
ORDER BY country, total_spent DESC
LIMIT 20;    